# Build Embeddings
This notebook generates embeddings for both SBERT (Sequence encoder) and OpenAI models and stores the indices inside the experiment folder.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [2]:

import pathlib, json, numpy as np, faiss
from tqdm import tqdm
from src.datasets.dataset import load_data
from src.rag.vector_store import VectorStore
from src.rag import _ARTIFACTS_DIR, _SBERT_DIR, _OPENAI_DIR
from src.embeddings.openai_embedder import OpenAIEmbedder
from sentence_transformers import SentenceTransformer

# ---- Parameters ----
N_CLASSES = int(os.getenv('N_CLASSES', '10'))
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OPENAI_MODEL = "text-embedding-3-small"

print('Artifacts root:', _ARTIFACTS_DIR)
_SBERT_DIR.mkdir(parents=True, exist_ok=True)
_OPENAI_DIR.mkdir(parents=True, exist_ok=True)


/home/marcmaceira/venv/reuters-rag-classifier/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Artifacts root: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts


In [3]:

# Load dataset
X_train, y_train, _, _, _ = load_data(n_classes=N_CLASSES)
print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


Loaded 6337 training documents with 10 classes


In [4]:

# ---- SBERT embeddings ----
sbert = SentenceTransformer(SBERT_MODEL)
vectors = sbert.encode(X_train, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype('float32')

meta = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, vectors)):
    meta.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

faiss_path = _SBERT_DIR / "index.faiss"
meta_path  = _SBERT_DIR / "meta.jsonl"
VectorStore.build(vectors, meta, vectors.shape[1], faiss_path, meta_path)
print("✅ SBERT index saved at", faiss_path)


2025-05-04 01:26:47,326 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2025-05-04 01:26:47,327 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Batches: 100%|██████████| 100/100 [03:09<00:00,  1.90s/it]
2025-05-04 01:29:58,951 - src.rag.vector_store - INFO - Building FAISS index with 6337 documents of dimension 384
2025-05-04 01:29:58,952 - src.rag.vector_store - INFO - Creating IndexFlatIP...
2025-05-04 01:29:58,956 - src.rag.vector_store - INFO - Normalizing embeddings...
2025-05-04 01:29:58,958 - src.rag.vector_store - INFO - Adding embeddings to index...
2025-05-04 01:29:58,963 - src.rag.vector_store - INFO - Added embeddings to index in 0.00 seconds
2025-05-04 01:29:58,964 - src.rag.vector_store - INFO - Writing index to /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/index.faiss...
2025-05-04 01:29:58,983 - src.rag.vector_

✅ SBERT index saved at /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/index.faiss


In [5]:

# ---- OpenAI embeddings ----
# Requires OPENAI_API_KEY env var
openai_embedder = OpenAIEmbedder(model=OPENAI_MODEL, batch_size=50)
openai_vecs = openai_embedder.encode(X_train)
openai_vecs = np.array(openai_vecs, dtype='float32')

meta_openai = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, openai_vecs)):
    meta_openai.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

openai_faiss = _OPENAI_DIR / "index.faiss"
openai_meta  = _OPENAI_DIR / "meta.jsonl"
VectorStore.build(openai_vecs, meta_openai, openai_vecs.shape[1], openai_faiss, openai_meta)
print("✅ OpenAI index saved at", openai_faiss)


2025-05-04 01:33:07,330 - src.embeddings.openai_embedder - INFO - Starting OpenAI embedding generation for 6337 texts with model text-embedding-3-small
2025-05-04 01:33:07,332 - src.embeddings.openai_embedder - INFO - Processing embedding batch 1 with 50 texts
2025-05-04 01:33:08,308 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-04 01:33:09,109 - src.embeddings.openai_embedder - INFO - Batch 1 used 8422 tokens
2025-05-04 01:33:09,110 - src.embeddings.openai_embedder - INFO - Batch 1 completed in 1.78 seconds
2025-05-04 01:33:09,111 - src.embeddings.openai_embedder - INFO - Average time per text in batch: 0.0356 seconds
2025-05-04 01:33:09,113 - src.embeddings.openai_embedder - INFO - Processing embedding batch 2 with 50 texts
2025-05-04 01:33:09,949 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-04 01:33:10,652 - src.embeddings.openai_embedder - INFO - Batch 2 used 7517 tokens
2025-

✅ OpenAI index saved at /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/openai/index.faiss
